# k-NN with Distance Metrics - Customer Segmentation Lab

## Business Scenario
You work as a data scientist for RetailIQ, an e-commerce analytics company. One of your clients, an online fashion retailer, wants to segment its customers for targeted marketing.

Five segments were identified in earlier research:
- Segment 0: Occasional Shoppers (low frequency, low value)
- Segment 1: Loyal Regular Shoppers (high frequency, moderate value)
- Segment 2: High-Value Enthusiasts (high frequency, high value)
- Segment 3: Big Spenders (low frequency, high value)
- Segment 4: New Customers (recent first purchase)

**Goal:** classify customers into these segments with k-NN, comparing distance metrics.

**Before running:** upload `retail_customer_data.csv` to the Colab session (Files panel on the left).

## Step 0: Setup - Import Libraries and Load Data

In [ ]:
# CodeGrade step0
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# OPTIONAL demo-data helper: does nothing if retail_customer_data.csv is already uploaded.
# If the file is missing, it creates a SYNTHETIC stand-in so the notebook can run end to end.
# Delete this cell and use the real file before you submit.
import os
if not os.path.exists('retail_customer_data.csv'):
    print('WARNING: retail_customer_data.csv not found - generating synthetic demo data.')
    rng = np.random.default_rng(42)
    n = 1000
    seg = rng.choice(5, n, p=[.25, .25, .15, .15, .20])
    prof = {  # recency, frequency, monetary, tenure, visits, mins, wishlist, abandons, age, income
        0: [60, 3, 120, 400, 8, 12, 2, 3, 38, 45000],
        1: [25, 20, 450, 700, 25, 35, 5, 2, 42, 65000],
        2: [10, 28, 1500, 800, 35, 50, 8, 1, 40, 110000],
        3: [45, 4, 1400, 500, 10, 18, 3, 2, 45, 120000],
        4: [8, 2, 90, 30, 15, 20, 4, 4, 30, 50000]}
    rows = []
    for s in seg:
        p = np.array(prof[s], dtype=float)
        v = p * (1 + rng.normal(0, .35, 10))
        rows.append(np.abs(v))
    d = pd.DataFrame(rows, columns=['recency','frequency','monetary','tenure','website_visits',
                                    'time_spent_mins','wishlist_items','cart_abandons','age','income'])
    for c in ['frequency','website_visits','wishlist_items','cart_abandons','age']:
        d[c] = d[c].round().astype(int)
    d['segment'] = seg
    d.to_csv('retail_customer_data.csv', index=False)

### Loading and exploring the dataset
The dataset contains customer information and a target variable `segment` (0-4).

In [ ]:
# CodeGrade step0
# Load the dataset
customer_data = pd.read_csv('retail_customer_data.csv')

In [ ]:
# Run this cell without changes
# Display basic information about the dataset
customer_data.info()

In [ ]:
# Run this cell without changes
# Check the first few rows of the dataset
customer_data.head()

In [ ]:
# Run this cell without changes
# Check the distribution of the target variable
customer_data['segment'].value_counts(normalize=True)

In [ ]:
# Run this cell without changes
# Check basic statistics of the dataset
customer_data.describe()

## Part 1: Analyzing Dataset Characteristics
Before selecting a distance metric, we look at feature distributions and correlations.

### Feature Distributions

In [ ]:
# CodeGrade step1
# Create histograms of all features to observe their distributions
# Select all numeric columns except the target
numeric_columns = customer_data.select_dtypes(include=[np.number]).columns.drop('segment')

# Plot histograms
plt.figure(figsize=(15, 10))
customer_data[numeric_columns].hist(bins=20, figsize=(15, 10))
plt.tight_layout()
plt.show();

# Create a correlation matrix to identify feature relationships - use the full dataframe including segment
correlation_matrix = customer_data.corr()

# Plot the correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show();

### Data Characteristics Analysis
**Scales.** The features are on very different scales: `income` and `monetary` run into the thousands or tens of thousands, while `frequency`, `wishlist_items` and `cart_abandons` are small counts. Without scaling, distance calculations are dominated by the large-valued features and the small ones barely matter.

**Correlations.** The heatmap shows which features move together (for example spending and frequency measures). Correlated features effectively count the same information twice in Euclidean and Manhattan distance. Mahalanobis distance accounts for the covariance between features, so it is the natural candidate when correlations are strong.

**Preprocessing.** Split into train and test sets first, then standardize with `StandardScaler` fit on the training data only, so every feature contributes equally and no test information leaks into training.

*(Check these statements against your own plots and edit if your data shows something different.)*

## Part 2: Data Preprocessing
Prepare the data for k-NN modeling.

In [ ]:
# CodeGrade step2
# Prepare features (X) and target (y)
X = customer_data.drop('segment', axis=1)
y = customer_data['segment']

# Splitting the data into training and testing sets (75-25 split and random_state of 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Run this cell without changes to display results
# Display a comparison of original vs. scaled data for the first sample
print("Original first sample:")
print(X_train.iloc[0].values[:5], "...")
print("\nScaled first sample:")
print(X_train_scaled[0][:5], "...")

## Part 3: Implementing k-NN with Different Distance Metrics
Evaluate each metric with 5-fold cross-validation accuracy.

In [ ]:
# CodeGrade step3
# Distance metrics to test
metrics = ['euclidean', 'manhattan', 'chebyshev']
k_value = 5

# Dictionary to store results
results = {}

# Loop through list of metrics
for metric in metrics:
    # Create and evaluate model with different metrics and k=5
    knn = KNeighborsClassifier(n_neighbors=k_value, metric=metric)
    # Get cross val scores for model
    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    # Store the mean of cv scores as value and metric name as key in results dictionary
    results[metric] = cv_scores.mean()

best_metric = max(results, key=results.get)

In [ ]:
# Run this cell without changes
# Find the best metric
print(results)
print(f"\nBest metric: {best_metric} with accuracy: {results[best_metric]:.4f}")

## Part 4: Implementing Mahalanobis Distance
Mahalanobis distance is useful when features are correlated. scikit-learn needs the **inverse** covariance matrix (`VI`).

In [ ]:
# CodeGrade step4
# Calculate covariance matrix from the training data
cov = np.cov(X_train_scaled, rowvar=False)

# Implement k-NN with Mahalanobis distance and k=5
knn_mahalanobis = KNeighborsClassifier(n_neighbors=5, metric='mahalanobis',
                                       metric_params={'VI': np.linalg.inv(cov)},
                                       algorithm='brute')

# Evaluate performance via cross validation
cv_scores_mahalanobis = cross_val_score(knn_mahalanobis, X_train_scaled, y_train, cv=5)
results['mahalanobis'] = cv_scores_mahalanobis.mean()

In [ ]:
# Run this cell without changes
print(f"Average CV accuracy with Mahalanobis: {cv_scores_mahalanobis.mean():.4f}")

# Update best metric if necessary
if results['mahalanobis'] > results[best_metric]:
    best_metric = 'mahalanobis'
    print(f"New best metric: {best_metric} with accuracy: {results[best_metric]:.4f}")

## Part 5: Hyperparameter Tuning
Grid search for the best k and weighting scheme with the top-performing metric:
- `n_neighbors`: [1, 3, 5, 7, 9]
- `weights`: ['uniform', 'distance']

In [ ]:
# CodeGrade step5
# Define parameter grid
param_grid = {'n_neighbors': [1, 3, 5, 7, 9],
              'weights': ['uniform', 'distance']}

# Create base model with best metric
if best_metric == 'mahalanobis':
    base_model = KNeighborsClassifier(metric='mahalanobis',
                                      metric_params={'VI': np.linalg.inv(cov)},
                                      algorithm='brute')
else:
    base_model = KNeighborsClassifier(metric=best_metric)

# Initialize and run grid search
grid_search = GridSearchCV(base_model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

# Get best parameters and accuracy
best_params = grid_search.best_params_
best_cv_accuracy = grid_search.best_score_

In [ ]:
# Run this cell without changes
# Visualization of accuracy for different k values
print(f"Best parameters: {best_params}")
print(f"Best cross-validation accuracy: {best_cv_accuracy:.4f}")

k_range = range(1, 31)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    k_scores.append(scores.mean())

plt.figure(figsize=(10, 6))
plt.plot(k_range, k_scores)
plt.xlabel('Value of k')
plt.ylabel('Cross-Validated Accuracy')
plt.title(f'Accuracy for Different Values of k using {best_metric}')
plt.grid(True)
plt.show()

## Part 6: Final Model Evaluation
Evaluate the tuned model on the held-out test set.

In [ ]:
# CodeGrade step6
# Build final model with best parameters
final_model = grid_search.best_estimator_

# Make predictions on test set
y_pred = final_model.predict(X_test_scaled)

# Calculate accuracy on test set
test_accuracy = accuracy_score(y_test, y_pred)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

In [ ]:
# Run this cell without changes
print(f"Test set accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()